# phonex GPU Indexer - Colab

## Progress Tracker
- ✅ Step 1: Check GPU
- ⏭️  Step 2: Upload phonex.zip via Google Drive
- ⏭️  Step 3: Extract ZIP
- ⏭️  Step 4: Install packages
- ⏭️  Step 5: Run indexer (GPU)
- ⏭️  Step 6: Download results

---

**IMPORTANT**: Before running, set GPU in Colab:
- Go to `Runtime` → `Change runtime type` → Select `GPU`

## ✅ Step 1: Check GPU

In [19]:
try:
    import torch
except ModuleNotFoundError:
    print("torch not found — installing...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

print("GPU Check:")
print(f"GPU Available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✅ GPU is READY")
else:
    print("\n❌ NO GPU! Go to Runtime > Change runtime type > GPU")


GPU Check:
GPU Available? True
GPU Name: Tesla T4
GPU Memory: 15.6 GB

✅ GPU is READY


## ⏭️  Step 2: Upload files via Google Drive

**Do this on your computer BEFORE running the cell below:**
1. Go to https://drive.google.com → open (or create) the folder **`My Drive/myph/`**
2. Upload **both** files into that folder:
   - `phonex.zip` — the indexer tool
   - your codebase zip, e.g. `aspnetcore-main.zip` — the code you want to index
3. Then run the cell below to copy them into Colab

> ⚠️ **Important:** The cell copies ALL `.zip` files from `My Drive/myph/` into `/content/`.  
> Do NOT put unrelated zips in that folder.


In [20]:
print("Mounting Google Drive...")
from google.colab import drive

drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")
print("\nSearching for ZIP files in Google Drive...")

import os, shutil

# clear out any existing zip files in the working directory so we start fresh
for existing in os.listdir('/content'):
    if existing.lower().endswith('.zip'):
        try:
            os.remove(os.path.join('/content', existing))
            print(f"   🗑️ removed old zip {existing}")
        except Exception as ex:
            print(f"   ⚠️ could not remove {existing}: {ex}")

# specify the Drive folder where you put the zip(s)
drive_folder = '/content/drive/My Drive/myph'

try:
    drive_files = os.listdir(drive_folder)
except FileNotFoundError:
    drive_files = []
    print(f"\n⚠️ Drive folder not found: {drive_folder}")

zips = [f for f in drive_files if f.endswith('.zip')]

if zips:
    print(f"\n✅ Found ZIP files: {zips}")
    print("\nCopying to Colab...")
    for z in zips:
        src = os.path.join(drive_folder, z)
        dst = f'/content/{z}'
        if not os.path.exists(dst):
            try:
                shutil.copy(src, dst)
                print(f"   ✅ Copied {z}")
            except FileNotFoundError:
                print(f"   ⚠️ Source not found: {src}")
    print("\n✅ Ready for extraction!")
else:
    print("\n❌ No ZIP files found in Google Drive")
    print("Upload phonex.zip to drive.google.com first!")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive mounted!

Searching for ZIP files in Google Drive...

✅ Found ZIP files: ['phonex.zip']

Copying to Colab...
   ✅ Copied phonex.zip

✅ Ready for extraction!


## ⏭️  Step 3: Extract ZIP File

In [21]:
import zipfile
import os
import subprocess

print("Searching for ZIP files...")

# clear out any existing extracted directories (anything not starting with '.' or 'drive' etc)
for item in os.listdir('/content'):
    path = os.path.join('/content', item)
    if os.path.isdir(path) and not item.startswith('.') and item not in ('drive', 'sample_data'):
        try:
            print(f"   🧹 removing old folder {item}/")
            subprocess.run(['rm', '-rf', path])
        except Exception:
            pass

# helper that will copy from drive if we haven't got any zips yet

def copy_from_drive_if_needed():
    drive_folder = '/content/drive/My Drive/myph'
    if os.path.exists(drive_folder):
        for f in os.listdir(drive_folder):
            if f.lower().endswith('.zip'):
                src = os.path.join(drive_folder, f)
                dst = os.path.join('/content', f)
                if not os.path.exists(dst):
                    try:
                        shutil.copy(src, dst)
                        print(f"   🔁 auto‑copied {f} from Drive")
                    except Exception as ex:
                        print(f"   ⚠️ failed to copy {f}: {ex}")

# perform initial search
result = subprocess.run(['find', '/content/', '-maxdepth', '1', '-name', '*.zip', '-type', 'f'], 
                       capture_output=True, text=True)
zip_files = [f for f in result.stdout.strip().split('\n') if f]

# if nothing, try grabbing from drive folder automatically
if not zip_files:
    copy_from_drive_if_needed()
    result = subprocess.run(['find', '/content/', '-maxdepth', '1', '-name', '*.zip', '-type', 'f'], 
                       capture_output=True, text=True)
    zip_files = [f for f in result.stdout.strip().split('\n') if f]

if zip_files:
    zip_path = zip_files[0]
    print(f"✅ Found ZIP: {zip_path}")
    print(f"Extracting...")
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    
    print("✅ Extraction done!")
    os.remove(zip_path)
    print("✅ Removed ZIP file")
    
    print("\n📁 Extracted folders:")
    for item in sorted(os.listdir('/content/')):
        path = f'/content/{item}'
        if os.path.isdir(path) and not item.startswith('.'):
            print(f"   📁 {item}/")
else:
    print("❌ No ZIP files found in /content/")
    print("Run Step 2 first to copy from Google Drive!")

Searching for ZIP files...
   🧹 removing old folder phonex/
✅ Found ZIP: /content/phonex.zip
Extracting...
✅ Extraction done!
✅ Removed ZIP file

📁 Extracted folders:
   📁 drive/
   📁 phonex/
   📁 sample_data/


## ⏭️  Step 4: Install Dependencies

In [ ]:
print("Installing packages for GPU indexing...\n")
!pip install -q llama-index "chromadb==0.5.23" sentence-transformers llama-index-vector-stores-chroma llama-index-embeddings-huggingface jedi
print("\n[OK] All packages installed!")


Installing packages for GPU indexing...


✅ All packages installed!


## ⏭️  Step 5: Run Indexer (GPU Processing)

**This will take 15 minutes to hours depending on your codebase size. Be patient!**

### 🔢 Execution Order (RUN IN THIS ORDER):

1. **Cell 1️⃣** — Verification & Diagnostics  
   → Checks for issues, shows what sources will be indexed  
   → RUN THIS FIRST

2. **Cell 2️⃣** — Main Indexer (GPU Processing)  
   → Actually runs the indexer  
   → RUN THIS SECOND (only after Cell 1 shows ✅ INDEXER WILL USE)


In [24]:
## ✅ [STEP 5 - CELL 1️⃣] Pre-flight Check & Auto-patch (RUN THIS FIRST)

import os, re

print("=" * 70)
print("🔍 STEP 5 - CELL 1️⃣: PRE-FLIGHT CHECK")
print("=" * 70)

# ── 1. Scan for all indexable sources ────────────────────────────────────
print("\n📁 Scanning for indexable sources...\n")

SYSTEM_SKIP = {'.config', 'drive', 'sample_data', '__pycache__', '__MACOSX', 'phonex'}
sources = []

# Top-level /content/
for item in sorted(os.listdir('/content/')):
    path = f'/content/{item}'
    if item.lower().endswith('.zip') and os.path.isfile(path):
        sources.append(path)
        print(f"   ✅ /content/{item}  ← zip, will be indexed")
    elif os.path.isdir(path) and not item.startswith('.') and item not in SYSTEM_SKIP:
        sources.append(path)
        print(f"   ✅ /content/{item}/  ← dir, will be indexed")
    else:
        ext = os.path.splitext(item)[1].lower()
        if item in SYSTEM_SKIP:
            reason = "system/tool dir"
        elif item.startswith('.'):
            reason = "hidden"
        else:
            reason = f"{ext or 'no ext'} — not indexable"
        print(f"   ⏭️  /content/{item}  ({reason})")

# Inside /content/phonex/ — zips (e.g. aspnetcore-main.zip) and data/ folder
phonex_sub = '/content/phonex'
if os.path.isdir(phonex_sub):
    for item in sorted(os.listdir(phonex_sub)):
        path = os.path.join(phonex_sub, item)
        if item.lower().endswith('.zip') and os.path.isfile(path):
            sources.append(path)
            print(f"   ✅ /content/phonex/{item}  ← zip (bundled in phonex/), will be indexed")
        elif item == 'data' and os.path.isdir(path):
            sources.append(path)
            print(f"   ✅ /content/phonex/data/  ← data folder, will be indexed")

print()
if sources:
    print(f"✅ Found {len(sources)} source(s): {[os.path.basename(s) for s in sources]}")
else:
    print("❌ NO INDEXABLE SOURCES FOUND")
    print()
    print("   Options:")
    print("   A) Put your codebase zip (e.g. aspnetcore-main.zip) in My Drive/myph/")
    print("      alongside phonex.zip, re-run Step 2, then re-run this cell.")
    print("   B) Put files in the phonex/data/ folder before zipping phonex.zip,")
    print("      re-run Step 3, then re-run this cell.")
    raise SystemExit("⛔ Stopping — nothing to index.")

# ── 2. Force-patch index_codebase.py with latest detection logic ─────────
indexer_path = '/content/phonex/index_codebase.py'
print(f"\n🔧 Patching {indexer_path}...")

with open(indexer_path, 'r') as f:
    content = f.read()

# Ensure global declaration
if 'global CODEBASE_DIRS' not in content:
    content = content.replace('def main():\n', 'def main():\n    global CODEBASE_DIRS\n', 1)
    print("   ✓ Added 'global CODEBASE_DIRS'")

NEW_DETECT_BLOCK = '''    if IN_COLAB:
        print("      🔍 Colab mode: auto-detecting codebase...")
        detected = []
        script_dir = os.path.realpath(os.path.dirname(os.path.abspath(__file__)))
        system_dirs = {'drive', 'sample_data', '__pycache__', '__MACOSX'}
        try:
            items = sorted(os.listdir('/content/'))
            for item in items:
                path = f'/content/{item}'
                if item.lower().endswith('.zip') and os.path.isfile(path):
                    detected.append(path)
                    print(f"        📦 zip: {item}")
            phonex_sub = '/content/phonex'
            if os.path.isdir(phonex_sub):
                for sub_item in sorted(os.listdir(phonex_sub)):
                    sub_path = os.path.join(phonex_sub, sub_item)
                    if sub_item.lower().endswith('.zip') and os.path.isfile(sub_path):
                        detected.append(sub_path)
                        print(f"        📦 zip (in phonex/): {sub_item}")
                data_dir = os.path.join(phonex_sub, 'data')
                if os.path.isdir(data_dir):
                    detected.append(data_dir)
                    print(f"        📁 data dir: phonex/data/")
            for item in items:
                path = f'/content/{item}'
                if (os.path.isdir(path) and not item.startswith('.')
                        and item not in system_dirs
                        and os.path.realpath(path) != script_dir):
                    detected.append(path)
                    print(f"        📁 dir:  {item}")
        except Exception as e:
            print(f"      ⚠️ Error scanning /content/: {e}")
        if detected:
            CODEBASE_DIRS = detected
            print(f"      ✓ Found and using {len(detected)} source(s):")
            for d in detected:
                print(f"        - {os.path.basename(d)}")
            print(f"      CODEBASE_DIRS is now: {CODEBASE_DIRS}")
        else:
            print(f"      ⚠️ No sources found, will try defaults")
            print(f"      Items in /content/: {os.listdir('/content/')}")'''

content = re.sub(
    r'    if IN_COLAB:.*?(?=\n    print\(f"\\n      📁 Will scan)',
    NEW_DETECT_BLOCK + '\n',
    content,
    count=1,
    flags=re.S,
)

with open(indexer_path, 'w') as f:
    f.write(content)
print("   ✓ Detection block patched (zips, phonex/data/, dirs all detected)")
print("\n✅ Pre-flight complete — proceed to run Cell 2️⃣ (Main Indexer)")


🔍 STEP 5 - CELL 1️⃣: PRE-FLIGHT CHECK

📁 Scanning for indexable sources...

   ⏭️  /content/.config  (system/tool dir)
   ⏭️  /content/Vanguard Software Architecture and Design (SAD)_new.doc  (.doc — not indexable)
   ⏭️  /content/drive  (system/tool dir)
   ⏭️  /content/phonex  (system/tool dir)
   ⏭️  /content/sample_data  (system/tool dir)
   ✅ /content/phonex/data/  ← data folder, will be indexed

✅ Found 1 source(s): ['data']

🔧 Patching /content/phonex/index_codebase.py...
   ✓ Detection block patched (zips, phonex/data/, dirs all detected)

✅ Pre-flight complete — proceed to run Cell 2️⃣ (Main Indexer)


In [25]:
## ✅ [STEP 5 - CELL 2️⃣] Main Indexer - GPU Processing (RUN THIS SECOND)

import os
import subprocess
import sys
import time

print("=" * 70)
print("🚀 STEP 5 - CELL 2️⃣: MAIN INDEXER (Running on GPU)")
print("=" * 70)

# Find the indexer script
indexer = None
if os.path.exists('/content/phonex/index_codebase.py'):
    indexer = '/content/phonex/index_codebase.py'
elif os.path.exists('/content/index_codebase.py'):
    indexer = '/content/index_codebase.py'

if indexer:
    print(f"Found indexer at: {indexer}")

    extracted_dirs = [d for d in os.listdir('/content/')
                     if os.path.isdir(f'/content/{d}') and not d.startswith('.')
                     and d not in ('drive', 'sample_data', '__pycache__')]
    print(f"📁 Extracted directories found: {extracted_dirs}\n")
    print("=" * 70)
    print("RUNNING INDEXER — live output below:")
    print("=" * 70)

    start = time.time()

    # Stream stdout and stderr in real time
    process = subprocess.Popen(
        [sys.executable, indexer],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )

    stderr_lines = []

    import threading

    def stream_stderr():
        for line in process.stderr:
            stderr_lines.append(line)
            # Only print non-progress-bar stderr lines to keep output clean
            stripped = line.rstrip()
            if stripped and not any(c in stripped for c in ['▏','▎','▍','▌','▋','▊','▉','█','▏']):
                if 'it/s' not in stripped and '\r' not in stripped:
                    print(f"[stderr] {stripped}", flush=True)

    t = threading.Thread(target=stream_stderr, daemon=True)
    t.start()

    for line in process.stdout:
        print(line, end='', flush=True)

    process.wait()
    t.join(timeout=5)
    elapsed = time.time() - start

    index_created = os.path.exists('/tmp/phonex_index') or os.path.exists('/content/phonex_index')

    print("\n" + "=" * 70)
    if process.returncode == 0 and index_created:
        print(f"✅ INDEXING COMPLETE in {elapsed:.1f} seconds ({elapsed/60:.1f} min)!")
    elif process.returncode == 0:
        print(f"⚠️ Script finished ({elapsed:.1f}s) but no index folder was found.")
        print("   → The codebase may have had no files, or an unexpected save path was used.")
    else:
        print(f"❌ Script failed with exit code {process.returncode} after {elapsed:.1f}s")
        print("   → Check the output above for error messages.")
        if stderr_lines:
            print("\n--- Full STDERR ---")
            print(''.join(stderr_lines))
    print("=" * 70)
else:
    print("ERROR: Could not find index_codebase.py")
    print("\nMake sure:")
    print("1. ZIP was extracted (run Step 3)")
    print("2. phonex folder is in /content/")


🚀 STEP 5 - CELL 2️⃣: MAIN INDEXER (Running on GPU)
Found indexer at: /content/phonex/index_codebase.py
📁 Extracted directories found: ['phonex']

RUNNING INDEXER — live output below:
phonex — Codebase Indexer

[1/4] Configuring embedding model...
[stderr] BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
[stderr] Key                     | Status     |  |
[stderr] ------------------------+------------+--+-
[stderr] embeddings.position_ids | UNEXPECTED |  |
[stderr] Notes:
[stderr] - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[stderr] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[stderr] 2026-02-27 15:32:12,175 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
      ✓ Embedder ready (model: all-MiniLM-L6-v2)
[2/4] Loading documents from codeb

## ⏭️  Step 6: Download Vector Database

In [27]:
import os
import shutil
import subprocess
from google.colab import files

# ── Find the index ────────────────────────────────────────────────────────
index_dir = None
search_paths = [
    '/tmp/phonex_index',
    '/content/phonex_index',
    '/content/phonex/phonex_index',
]
for path in search_paths:
    if os.path.exists(path):
        index_dir = path
        print(f"✅ Found index at: {index_dir}")
        break

if not index_dir:
    print("🔍 Searching for phonex_index folders...")
    for search_root in ['/tmp/', '/content/']:
        result = subprocess.run(
            ['find', search_root, '-name', 'phonex_index', '-type', 'd', '-maxdepth', '3'],
            capture_output=True, text=True)
        found = [p for p in result.stdout.strip().split('\n') if p]
        if found:
            index_dir = found[0]
            print(f"✅ Found index at: {index_dir}")
            break

if not index_dir or not os.path.exists(index_dir):
    print("❌ Index not found!")
    print("\n⚠️ Common reasons:")
    print("1. Step 5 (indexer) failed or didn't complete. Check its output above.")
    print("2. No source code files were found to index.")
    print("\n🔧 Re-run Step 5 and check for error messages.")
else:
    # ── Stats ─────────────────────────────────────────────────────────────
    total_size = sum(
        os.path.getsize(os.path.join(dp, fn))
        for dp, _, fns in os.walk(index_dir) for fn in fns
    )
    file_count = sum(len(fns) for _, _, fns in os.walk(index_dir))
    print(f"\n📊 Index Statistics:")
    print(f"   Size:  {total_size / 1024**2:.1f} MB")
    print(f"   Files: {file_count}")

    # ── Zip it ────────────────────────────────────────────────────────────
    zip_path = '/tmp/phonex_index_download.zip'
    print(f"\nZipping to {zip_path}...")
    shutil.make_archive('/tmp/phonex_index_download', 'zip', index_dir)
    print(f"✅ Zip created ({os.path.getsize(zip_path)/1024**2:.1f} MB)")

    # ── Save to Google Drive ──────────────────────────────────────────────
    drive_dest_dir = '/content/drive/My Drive/myph'
    drive_dest_file = os.path.join(drive_dest_dir, 'phonex_index_download.zip')
    if os.path.exists(drive_dest_dir):
        shutil.copy(zip_path, drive_dest_file)
        print(f"\n✅ Saved to Google Drive: My Drive/myph/phonex_index_download.zip")
        print(f"\n📋 On your local machine:")
        print(f"   1. Open Google Drive on your PC (drive.google.com or Drive app)")
        print(f"   2. Download  My Drive/myph/phonex_index_download.zip")
        print(f"   3. Extract to:  D:\\Gitrnd\\phonex\\indexfolder\\")
    else:
        print("\n⚠️ Google Drive not mounted — skipping Drive save.")
        print("   Run Step 2 first to mount Drive if you want to save there.")

    # ── Also trigger browser download as fallback ─────────────────────────
    print("\nTriggering browser download as well...")
    files.download(zip_path)
    print("\n✅ DONE! Check your browser downloads or Google Drive.")
    print(f"   Extract the zip to:  D:\\Gitrnd\\phonex\\indexfolder\\")


✅ Found index at: /tmp/phonex_index

📊 Index Statistics:
   Size:  438.5 MB
   Files: 22

Zipping to /tmp/phonex_index_download.zip...
✅ Zip created (146.4 MB)

✅ Saved to Google Drive: My Drive/myph/phonex_index_download.zip

📋 On your local machine:
   1. Open Google Drive on your PC (drive.google.com or Drive app)
   2. Download  My Drive/myph/phonex_index_download.zip
   3. Extract to:  D:\Gitrnd\phonex\indexfolder\

Triggering browser download as well...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ DONE! Check your browser downloads or Google Drive.
   Extract the zip to:  D:\Gitrnd\phonex\indexfolder\


## ✅ All Done!

### What you got:
- **phonex_index_download.zip** — your complete vector database (Chroma index)

### The zip was saved to two places:
1. **Google Drive** → `My Drive/myph/phonex_index_download.zip`
2. **Browser download** → your Downloads folder

### Extract it locally:
```
Destination: D:\Gitrnd\phonex\indexfolder\
```
Right-click the zip → Extract All → paste path above.

### Next steps on your local machine:
1. Point `server.py` / `test_query.py` at `D:\Gitrnd\phonex\indexfolder\`
2. Run queries — GPU not needed for querying!
